# Brandenburg-Höhenkacheln nach R2 (bDOM)

Prozessiert das **bildbasierte DOM (bDOM)** von Brandenburg flächendeckend zu den
binären Höhen-Kacheln der Elevation API und lädt sie nach Cloudflare R2.

**Warum bDOM statt ALS-LAZ?** Gleiche DOM-Semantik (Oberfläche inkl. Bäume/Gebäude),
aber ~5× kleiner (~22 MB statt ~114 MB pro Kachel) und ohne Punktwolken-Gridding.

**Auflösung:** bDOM ist nativ **0,2 m** (5000×5000 pro km-Kachel). Wir rechnen auf
**1 m** herunter (`Resampling.max` = höchster Punkt gewinnt, wie das alte DSM), damit das
Format byte-kompatibel bleibt und der Worker unverändert läuft. (Feiner ginge, würde aber
Tile-Größe/R2-Speicher vervielfachen und Worker-Anpassungen erfordern.)

**Format (byte-kompatibel zu den bestehenden Kacheln):** 1000×1000, 1 m, Uint16 in cm,
Little-Endian, Zeile 0 = Süden, Index `row*1000+col`, nodata = 0.

**Ablauf:** alle bDOM-Kacheln auflisten → bereits auf R2 vorhandene überspringen (Resume)
→ je Kachel: ZIP laden → GeoTIFF auf das km-Raster resampeln → spiegeln → `.bin`(+`.bin.gz`)
→ nach R2 hochladen.

**Maßstab ganz BB:** ~30k Kacheln, ~680 GB Download, ~60 GB `.bin` auf R2. Free-Colab
trennt nach ~12 h — einfach die Run-Zelle erneut starten (Resume überspringt Fertiges).

## Vorbereitung (einmalig)
1. Cloudflare → R2 → **Manage R2 API Tokens** → *Create API Token* (Object Read & Write,
   Bucket `windrad-tiles`). Du erhältst **Access Key ID** + **Secret Access Key**.
2. In Colab: 🔑 **Secrets** (linke Leiste) anlegen:
   `R2_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY` (Notebook-Zugriff aktivieren).
   `R2_ACCOUNT_ID` ist die Account-ID (Cloudflare-Dashboard-URL).

In [ ]:
!pip -q install rasterio boto3 requests tqdm numpy

In [ ]:
import os, io, re, zipfile, gzip, requests, numpy as np, rasterio
from rasterio.warp import reproject, Resampling, transform as warp_transform
from rasterio.transform import from_origin
from concurrent.futures import ThreadPoolExecutor, as_completed
import boto3
from botocore.config import Config

# --- R2-Zugang: bevorzugt aus Colab-Secrets, sonst aus Umgebungsvariablen ---
try:
    from google.colab import userdata
    def get(k):
        try: return userdata.get(k)
        except Exception: return ''
except Exception:
    def get(k): return os.environ.get(k, '')

R2_ACCOUNT_ID = get('R2_ACCOUNT_ID') or '975505fa80cf3d0f8e0c3b049e9c6112'
R2_ACCESS_KEY = get('R2_ACCESS_KEY_ID')
R2_SECRET_KEY = get('R2_SECRET_ACCESS_KEY')

BUCKET     = 'windrad-tiles'
# --- Quelle: Bundesland x Produkt. kind='dom' (Oberflaeche, fuer Sichtlinien)
#              kind='dgm' (Gelaende ohne Bewuchs/Bebauung, fuer Gefaelle/Aufstellflaechen)
PRESETS = {
    'BB_DOM':  dict(zone=33, kind='dom', base='https://data.geobasis-bb.de/geobasis/daten/bdom/tif',
                   name_re=r'bdom_33(\d+)-(\d+)\.zip'),
    'BB_DGM':  dict(zone=33, kind='dgm', base='https://data.geobasis-bb.de/geobasis/daten/dgm/tif',
                   name_re=r'dgm_33(\d+)-(\d+)\.zip'),
    'NRW_DOM': dict(zone=32, kind='dom', base='https://www.opengeodata.nrw.de/produkte/geobasis/hm/dom1_tiff/dom1_tiff',
                   name_re=r'dom1_32_(\d+)_(\d+)_1_nw_\d+\.tif'),
    'NRW_DGM': dict(zone=32, kind='dgm', base='https://www.opengeodata.nrw.de/produkte/geobasis/hm/dgm1_tiff/dgm1_tiff',
                   name_re=r'dgm1_32_(\d+)_(\d+)_1_nw_\d+\.tif'),
}
MODELS     = ['NRW_DOM', 'NRW_DGM']   # nacheinander abgearbeitet (Resume ueberspringt Fertiges)
ZONE = KIND = BDOM_BASE = NAME_RE = KEY_PREFIX = None
def apply_preset(name):
    global ZONE, KIND, BDOM_BASE, NAME_RE, KEY_PREFIX
    pr = PRESETS[name]
    ZONE, KIND, BDOM_BASE, NAME_RE = pr['zone'], pr['kind'], pr['base'], pr['name_re']
    KEY_PREFIX = 'tile' if KIND == 'dom' else 'dgm'   # DOM: tile_<zone>_E_N ; DGM: dgm_<zone>_E_N
apply_preset(MODELS[0])
TILE_SIZE  = 1000      # Meter pro Kachel
GRID       = 1000      # Zellen pro Kante -> 1 m Auflösung
UPLOAD_GZ  = True      # zusätzlich .bin.gz hochladen
WORKERS    = 8         # parallele Downloads/Uploads
# --- Gebiet: None = ganzes Land (hier ganz NRW). Sonst {'center':(lat,lon),'radius_km':n}
#     oder {'bbox':(Sued,West,Nord,Ost)}. BBOX wird je Modell/Zone berechnet.
AREA = None
def bbox_km(area, zone):
    if not area: return None
    epsg = 25832 if zone == 32 else 25833
    def u(lat, lon):
        xs, ys = warp_transform('EPSG:4326', f'EPSG:{epsg}', [lon], [lat])
        return xs[0] / 1000, ys[0] / 1000
    if 'center' in area:
        (la, lo), r = area['center'], area['radius_km']; cx, cy = u(la, lo)
        return (int(cx - r), int(cx + r) + 1, int(cy - r), int(cy + r) + 1)
    s, w, n, e = area['bbox']
    pts = [u(a, o) for a, o in ((s, w), (s, e), (n, w), (n, e))]
    ex = [p[0] for p in pts]; ny = [p[1] for p in pts]
    return (int(min(ex)), int(max(ex)) + 1, int(min(ny)), int(max(ny)) + 1)

# R2-Key trägt die UTM-Zone (Worker liest tile_<zone>_<E>_<N>)
def tile_key(tx, ty, ext='bin'):
    return f'{KEY_PREFIX}_{ZONE}_{tx}_{ty}.{ext}'

s3 = boto3.client('s3',
    endpoint_url=f'https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com',
    aws_access_key_id=R2_ACCESS_KEY, aws_secret_access_key=R2_SECRET_KEY,
    region_name='auto',
    config=Config(retries={'max_attempts': 5, 'mode': 'standard'}))
assert R2_ACCESS_KEY and R2_SECRET_KEY, 'R2-Secrets fehlen (siehe Markdown oben).'
print('R2-Client bereit. Account:', R2_ACCOUNT_ID[:8] + '…')

In [ ]:
# Kacheln des Portals auflisten. Wir merken den VOLLEN Dateinamen je (E,N),
# weil er je Land Suffixe/Jahr enthält (NRW: ..._1_nw_2022.tif).
def list_all_tiles():
    html = requests.get(BDOM_BASE + '/', timeout=180).text
    files = {}
    for m in re.finditer(NAME_RE, html):
        files[(int(m.group(1)), int(m.group(2)))] = m.group(0)
    return files

# (Aufruf erfolgt im Treiber, Zelle 6)

In [ ]:
# Resume: bereits als .bin auf R2 vorhandene Kacheln ermitteln
def list_done():
    done, token = set(), None
    while True:
        kw = {'Bucket': BUCKET, 'Prefix': f'{KEY_PREFIX}_{ZONE}_'}
        if token: kw['ContinuationToken'] = token
        r = s3.list_objects_v2(**kw)
        for o in r.get('Contents', []):
            m = re.match(rf'{KEY_PREFIX}_{ZONE}_(\d+)_(\d+)\.bin$', o['Key'])
            if m: done.add((int(m.group(1)), int(m.group(2))))
        if r.get('IsTruncated'): token = r['NextContinuationToken']
        else: break
    return done

# (Aufruf erfolgt im Treiber, Zelle 6)

In [ ]:
# GeoTIFF -> byte-kompatibles Uint16-cm-Grid (1000x1000, row0=Süden, nodata=0)
# Quelle bDOM ist 0.2 m (5000x5000) -> auf 1 m (1000x1000) heruntergerechnet.
def make_grid(tif_bytes, tx, ty):
    with rasterio.open(io.BytesIO(tif_bytes)) as src:
        dst = np.zeros((GRID, GRID), dtype=np.float32)
        # Ziel-Raster: north-up, NW-Ecke der km-Kachel, 1 m Auflösung
        dst_transform = from_origin(tx * TILE_SIZE, (ty + 1) * TILE_SIZE,
                                    TILE_SIZE / GRID, TILE_SIZE / GRID)
        reproject(
            source=rasterio.band(src, 1), destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=dst_transform, dst_crs=src.crs,
            src_nodata=src.nodata, dst_nodata=0.0,
            resampling=Resampling.max)   # hoechster Punkt gewinnt (wie DSM); erhaelt Hindernis-Hoehen
    dst = np.flipud(dst)                  # GeoTIFF ist north-up -> unser Format: row0=Süden
    dst[~np.isfinite(dst)] = 0.0
    dst[dst < 0] = 0.0
    return (dst * 100.0).astype('<u2')    # cm, little-endian Uint16

def process(tile):
    tx, ty = tile
    fname = TILEFILES[tile]
    r = requests.get(f'{BDOM_BASE}/{fname}', timeout=300); r.raise_for_status()
    if fname.lower().endswith('.zip'):
        zf = zipfile.ZipFile(io.BytesIO(r.content))
        inner = next(n for n in zf.namelist() if n.lower().endswith(('.tif', '.tiff')))
        tif_bytes = zf.read(inner)
    else:
        tif_bytes = r.content            # NRW & Co.: GeoTIFF direkt
    raw = make_grid(tif_bytes, tx, ty).tobytes()
    assert len(raw) == GRID * GRID * 2, f'falsche Größe {len(raw)}'
    s3.put_object(Bucket=BUCKET, Key=tile_key(tx, ty), Body=raw,
                  ContentType='application/octet-stream')
    if UPLOAD_GZ:
        s3.put_object(Bucket=BUCKET, Key=tile_key(tx, ty, 'bin.gz'),
                      Body=gzip.compress(raw), ContentType='application/gzip')
    return tile

In [ ]:
# TREIBER: arbeitet MODELS nacheinander ab (ganz NRW = AREA None). Resume-sicher.
# Bei Colab-Trennung: diese Zelle einfach erneut starten -> macht weiter.
import requests
NTFY = 'https://ntfy.sh/windrad-DEIN-GEHEIMES-WORT'   # eigenes Topic; in der ntfy-App abonnieren
def notify(msg, title='Hoehen-Pipeline'):
    try: requests.post(NTFY, data=msg.encode('utf-8'), headers={'Title': title}, timeout=10)
    except Exception: pass

for _pre in MODELS:
    apply_preset(_pre)
    _bb = bbox_km(AREA, ZONE)
    TILEFILES = list_all_tiles()          # global: process() liest hieraus den Dateinamen
    tiles = sorted(TILEFILES)
    if _bb:
        x0, x1, y0, y1 = _bb
        tiles = [(x, y) for (x, y) in tiles if x0 <= x <= x1 and y0 <= y <= y1]
    done = list_done()
    todo = [t for t in tiles if t not in done]
    print(f'[{_pre}] Portal {len(TILEFILES)} | Gebiet {len(tiles)} | schon da {len(done)} | zu tun {len(todo)}')
    notify(f'{_pre}: Start, {len(todo)} Kacheln')
    errors, ok = [], 0
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = {ex.submit(process, t): t for t in todo}
        for f in as_completed(futs):
            try: f.result(); ok += 1
            except Exception as e: errors.append((futs[f], str(e)))
    msg = f'{_pre} fertig: {ok} hochgeladen, {len(errors)} Fehler.'
    if errors: msg += ' z.B. ' + str(errors[0])
    print(msg); notify(msg)
print('ALLE MODELLE FERTIG:', MODELS)
notify('Alle Modelle fertig: ' + ', '.join(MODELS))

In [ ]:
# Plausibilitäts-Check einer Kachel aus R2 (aktives = zuletzt verarbeitetes Modell)
_ks = sorted(list_all_tiles()); tx, ty = _ks[len(_ks)//2]
raw = s3.get_object(Bucket=BUCKET, Key=tile_key(tx, ty))['Body'].read()
a = np.frombuffer(raw, dtype='<u2').reshape(GRID, GRID) / 100.0
v = a[a > 0]
print(f'{tile_key(tx,ty)}: min={v.min():.1f} mean={v.mean():.1f} max={v.max():.1f} m | '
      f'Abdeckung={100*v.size/a.size:.1f}%')

In [ ]:
# Abdeckungs-Manifest aus R2 erzeugen (robust: listet ALLE vorhandenen Kacheln, beide Zonen)
# -> lädt windrad-tiles.txt herunter; damit tiles/windrad-tiles.txt im Repo ersetzen.
def r2_manifest(bucket=BUCKET):
    keys, token = set(), None
    while True:
        kw = {'Bucket': bucket, 'Prefix': 'tile_'}
        if token: kw['ContinuationToken'] = token
        r = s3.list_objects_v2(**kw)
        for o in r.get('Contents', []):
            m = re.match(r'tile_(?:(\d+)_)?(\d+)_(\d+)\.bin$', o['Key'])
            if m:
                z = m.group(1) or '33'   # Legacy ohne Zone = 33
                keys.add(f'tile_{z}_{m.group(2)}_{m.group(3)}')
        if r.get('IsTruncated'): token = r['NextContinuationToken']
        else: break
    return sorted(keys)

_m = r2_manifest()
with open('windrad-tiles.txt', 'w') as f:
    f.write('\n'.join(_m) + '\n')
print(f'{len(_m)} Kacheln -> windrad-tiles.txt (ins Repo unter tiles/ legen)')
try:
    from google.colab import files; files.download('windrad-tiles.txt')
except Exception: pass